# Tencent WeMM-Embedding-9B + Qdrant — Kaggle T4×2 Production Demo

**VI:** Notebook public này giữ nguyên phần truy xuất semantic đã được chấp nhận và bổ sung một phần visual-retrieval riêng để chứng minh raw cosine `0.90+` một cách trung thực. Hai phần dùng **hai search space khác nhau** và được báo cáo riêng để tránh đánh đồng benchmark:

- **Semantic corpus retrieval:** Qdrant production corpus, `99,967` entities mỗi collection.
- **Visual robustness retrieval:** temporary curated gallery gồm `4` original images.

**EN:** This public notebook preserves the accepted semantic-retrieval showcase and adds a separate visual-retrieval section that demonstrates legitimate raw cosine `0.90+`. The two sections use **different search spaces** and are reported separately to avoid conflating benchmarks:

- **Semantic corpus retrieval:** production Qdrant corpus, `99,967` entities per collection.
- **Visual robustness retrieval:** temporary curated gallery containing `4` original images.

### Yêu cầu môi trường / Runtime requirements

- **Accelerator:** GPU T4 ×2
- **Internet:** ON
- **Dataset:** `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots`, version `1`
- **Model:** `dangkhoa2016/tencent-wemm-embedding-9b`, Transformers/default/version `1`

### Luồng trình diễn / Presentation flow

1. Steps 1–5 — system setup
2. Step 6 — bilingual text retrieval
3. Step 7A — semantic image→text cross-modal retrieval over the 99,967-entity corpus
4. Step 7B — transformed-image→original-image robustness retrieval over a 4-image temporary gallery
5. Step 8 — closeout, GPU reclaim, Qdrant storage seal


## Atomic modular runner / Trình chạy nguyên khối dạng module

**VI:** Kaggle vẫn chỉ submit **một code cell**, nhưng implementation dài đã được chuyển sang package `wemm_notebook/` để reviewer có thể đọc từng phase riêng. Code cell dưới đây chỉ checkout presentation source từ public Release ref, import `run_public_notebook()`, rồi chạy toàn bộ workflow nguyên khối. Frozen science/runtime authority vẫn được bootstrap riêng tại commit `d04bcd3e601b449b67d09ff1132cab965619d858`.

**EN:** Kaggle still submits **one code cell**, while the long implementation now lives in the readable `wemm_notebook/` package. The cell below only checks out presentation source from the public Release ref, imports `run_public_notebook()`, and executes the workflow atomically. Frozen science/runtime authority remains separately pinned to commit `d04bcd3e601b449b67d09ff1132cab965619d858`.

**Dependency authority:** the frozen runtime bootstrap installs both `requirements-kaggle.txt` and `requirements-demo.txt` from commit `d04bcd3e601b449b67d09ff1132cab965619d858`; the notebook presentation package remains sourced separately from the public release.


In [ ]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys

REPO = "https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU.git"
PUBLIC_RELEASE_REF = "v1.0.0"
SOURCE_ROOT = Path("/kaggle/working/wemm-public-notebook-source-v1.0.0")

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)

subprocess.run(["git", "init", "-q"], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "remote", "add", "origin", REPO], cwd=SOURCE_ROOT, check=True)
subprocess.run(
    ["git", "fetch", "-q", "--depth", "1", "origin", PUBLIC_RELEASE_REF],
    cwd=SOURCE_ROOT,
    check=True,
)
subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=SOURCE_ROOT, check=True)

sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

import wemm_notebook
from wemm_notebook import run_public_notebook

presentation_file = Path(wemm_notebook.__file__).resolve()
assert SOURCE_ROOT.resolve() in presentation_file.parents, (presentation_file, SOURCE_ROOT)
print("PUBLIC_NOTEBOOK_PRESENTATION_SOURCE=PASS", flush=True)
print("PUBLIC_NOTEBOOK_PRESENTATION_REF=" + PUBLIC_RELEASE_REF, flush=True)

run_public_notebook()
